In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Function Vectors Replication Notebook

This notebook replicates the key experiments from the "Function Vectors in Large Language Models" paper (Todd et al., ICLR 2024).

## Overview

The paper investigates whether autoregressive transformer language models develop compact vector representations of input-output functions (called function vectors) within their hidden states during in-context learning.

## Key Hypothesis:
A small number of attention heads transport a compact representation of the demonstrated task (function vector) that is robust to changes in context and can trigger task execution in zero-shot and natural text settings.

## Note on Replication
Since GPT-J 6B requires downloading ~24GB, we will use the repository's demo notebook approach and verify the functionality using the existing code infrastructure.

In [2]:
# Core imports and setup
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
from typing import Dict, List, Tuple, Optional, Any
from sklearn.model_selection import train_test_split
from pathlib import Path

# Add repository to path
repo_root = '/net/scratch2/smallyan/function_vectors_eval'
sys.path.insert(0, repo_root)

# Disable gradient computation for inference
torch.set_grad_enabled(False)

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Set seed for reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(0)
print("Seed set to 0")

Using device: cuda
GPU: NVIDIA H200 NVL
Seed set to 0


In [3]:
# Import from the repository's utilities
from src.utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
from src.utils.intervention_utils import fv_intervention_natural_text, function_vector_intervention
from src.utils.model_utils import load_gpt_model_and_tokenizer
from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
from src.utils.eval_utils import decode_to_vocab, sentence_eval

print("Repository utilities imported successfully!")

Repository utilities imported successfully!


## Load Model and Tokenizer

Load GPT-J 6B model - the smallest model used in the original experiments.

In [4]:
# Load the model
model_name = 'EleutherAI/gpt-j-6b'
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
EDIT_LAYER = 9  # Optimal layer for FV intervention (approximately L/3)

print(f"\nModel config:")
print(f"  Layers: {model_config['n_layers']}")
print(f"  Heads: {model_config['n_heads']}")
print(f"  Hidden dim: {model_config['resid_dim']}")
print(f"\nEdit layer for FV intervention: {EDIT_LAYER}")

Loading:  EleutherAI/gpt-j-6b


OSError: PermissionError at /net/projects2/chai-lab/shared_models/hub/.locks/models--EleutherAI--gpt-j-6b/0e183edc2025ecfdba4429ba43c960224103b3c3dc26616503cdc2158a3d6c93.lock when downloading EleutherAI/gpt-j-6b. Check cache directory permissions. Common causes: 1) another user is downloading the same model (please wait); 2) a previous download was canceled and the lock file needs manual removal.

In [5]:
# Set custom cache directory to avoid permission issues
import os
os.environ['HF_HOME'] = '/net/scratch2/smallyan/.cache/huggingface'
os.environ['TRANSFORMERS_CACHE'] = '/net/scratch2/smallyan/.cache/huggingface'
os.makedirs('/net/scratch2/smallyan/.cache/huggingface', exist_ok=True)

# Try loading model with custom cache
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading GPT-J 6B model with custom cache directory...")
model_name = 'EleutherAI/gpt-j-6b'

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir='/net/scratch2/smallyan/.cache/huggingface')
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    low_cpu_mem_usage=True,
    cache_dir='/net/scratch2/smallyan/.cache/huggingface',
    torch_dtype=torch.float16
).to('cuda')

print("Model loaded successfully!")

Loading GPT-J 6B model with custom cache directory...


`torch_dtype` is deprecated! Use `dtype` instead!
